In [1]:
import torch
import os, json, csv
from langchain_core.documents import Document
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader, DirectoryLoader
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from datetime import datetime
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_text_splitters import Language
import tiktoken

/home/oncreative/.local/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/oncreative/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
MODEL_ID = "google/gemma-3-1b-it"
pipe = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    device=0,
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": True,
        "temperature": 0.7,
        "return_full_text": False,
    },
)
llm = ChatHuggingFace(llm=pipe)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/home/oncreative/.local/lib/python3.10/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [3]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [7]:
markdown_text = """# 머신러닝 완벽 가이드

## 1장: 지도학습

### 1.1 분류 (Classification)
분류는 입력 데이터를 미리 정의된 카테고리로 분류하는 작업입니다.
대표적인 알고리즘으로 로지스틱 회귀, 결정 트리, SVM, 랜덤 포레스트가 있습니다.
성능 평가에는 정확도, 정밀도, 재현율, F1 스코어를 사용합니다.

### 1.2 회귀 (Regression)
회귀는 연속적인 수치를 예측하는 작업입니다.
선형 회귀, 다항 회귀, 릿지/라쏘 회귀가 대표적입니다.
평가 지표로는 MSE, RMSE, MAE, R² 스코어를 사용합니다.

## 2장: 비지도학습

### 2.1 군집화 (Clustering)
군집화는 레이블 없이 데이터를 그룹화하는 기법입니다.
K-Means, DBSCAN, 계층적 군집화가 주로 사용됩니다.

### 2.2 차원 축소 (Dimensionality Reduction)
고차원 데이터를 저차원으로 변환하여 시각화와 분석을 용이하게 합니다.
PCA, t-SNE, UMAP이 널리 사용됩니다.

## 3장: 딥러닝

딥러닝은 다층 신경망을 사용한 머신러닝의 한 분야입니다.
CNN, RNN, Transformer 등 다양한 아키텍처가 존재합니다.
"""


In [5]:
headers_to_split_on = [
    ('#', '대제목'),
    ('##', '장'),
    ('###', '절')
]

md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on = headers_to_split_on)
md_docs = md_splitter.split_text(markdown_text)

In [6]:
md_docs

[Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '1장: 지도학습', '절': '1.1 분류 (Classification)'}, page_content='분류는 입력 데이터를 미리 정의된 카테고리로 분류하는 작업입니다.\n대표적인 알고리즘으로 로지스틱 회귀, 결정 트리, SVM, 랜덤 포레스트가 있습니다.\n성능 평가에는 정확도, 정밀도, 재현율, F1 스코어를 사용합니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '1장: 지도학습', '절': '1.2 회귀 (Regression)'}, page_content='회귀는 연속적인 수치를 예측하는 작업입니다.\n선형 회귀, 다항 회귀, 릿지/라쏘 회귀가 대표적입니다.\n평가 지표로는 MSE, RMSE, MAE, R² 스코어를 사용합니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '2장: 비지도학습', '절': '2.1 군집화 (Clustering)'}, page_content='군집화는 레이블 없이 데이터를 그룹화하는 기법입니다.\nK-Means, DBSCAN, 계층적 군집화가 주로 사용됩니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '2장: 비지도학습', '절': '2.2 차원 축소 (Dimensionality Reduction)'}, page_content='고차원 데이터를 저차원으로 변환하여 시각화와 분석을 용이하게 합니다.\nPCA, t-SNE, UMAP이 널리 사용됩니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '3장: 딥러닝'}, page_content='딥러닝은 다층 신경망을 사용한 머신러닝의 한 분야입니다.\nCNN, RNN, Transformer 등 다양한 아키텍처가 존재합니다.')]

In [8]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
final_docs = text_splitter.split_documents(md_docs)

In [9]:
final_docs

[Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '1장: 지도학습', '절': '1.1 분류 (Classification)'}, page_content='분류는 입력 데이터를 미리 정의된 카테고리로 분류하는 작업입니다.\n대표적인 알고리즘으로 로지스틱 회귀, 결정 트리, SVM, 랜덤 포레스트가 있습니다.\n성능 평가에는 정확도, 정밀도, 재현율, F1 스코어를 사용합니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '1장: 지도학습', '절': '1.2 회귀 (Regression)'}, page_content='회귀는 연속적인 수치를 예측하는 작업입니다.\n선형 회귀, 다항 회귀, 릿지/라쏘 회귀가 대표적입니다.\n평가 지표로는 MSE, RMSE, MAE, R² 스코어를 사용합니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '2장: 비지도학습', '절': '2.1 군집화 (Clustering)'}, page_content='군집화는 레이블 없이 데이터를 그룹화하는 기법입니다.\nK-Means, DBSCAN, 계층적 군집화가 주로 사용됩니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '2장: 비지도학습', '절': '2.2 차원 축소 (Dimensionality Reduction)'}, page_content='고차원 데이터를 저차원으로 변환하여 시각화와 분석을 용이하게 합니다.\nPCA, t-SNE, UMAP이 널리 사용됩니다.'),
 Document(metadata={'대제목': '머신러닝 완벽 가이드', '장': '3장: 딥러닝'}, page_content='딥러닝은 다층 신경망을 사용한 머신러닝의 한 분야입니다.\nCNN, RNN, Transformer 등 다양한 아키텍처가 존재합니다.')]

In [10]:
import re
from typing import List
from langchain_text_splitters import TextSplitter

class RegexSplitter(TextSplitter):
    def __init__(self, pattern=r"제\d+장", **kwargs):
        super().__init__(**kwargs)
        self.pattern = pattern

    def split_text(self, text):
        splits = re.split(f"({self.pattern})", text)

        chunks = []
        current = ""
        for part in splits:
            if re.match(self.pattern, part):
                if current.strip():
                    chunks.append(current.strip())
                current = part
            else:
                current += part
        if current.strip():
            chunks.append(current.strip())
        return chunks

In [14]:
text = open('sample_docs/ml_textbook.txt', encoding='utf-8').read()
text[:30]

'제1장: 머신러닝 기초 개념 - 파트 1\n\n머신러닝의 '

In [16]:
splitter = RegexSplitter(pattern = r"제\d+장")
chunks = splitter.split_text(text)
chunks[:3]

['제1장: 머신러닝 기초 개념 - 파트 1\n\n머신러닝의 제1번째 핵심 개념을 다룹니다.\n지도학습, 비지도학습, 강화학습은 머신러닝의 세 가지 주요 패러다임입니다.\n모델의 성능은 학습 데이터의 품질과 양에 크게 의존합니다.\n과적합(Overfitting)을 방지하기 위해 정규화 기법이 사용됩니다.\n교차 검증(Cross-validation)은 모델의 일반화 성능을 평가하는 데 유용합니다.\n하이퍼파라미터 튜닝은 모델 최적화의 핵심 단계입니다.',
 '제2장: 머신러닝 기초 개념 - 파트 2\n\n머신러닝의 제2번째 핵심 개념을 다룹니다.\n지도학습, 비지도학습, 강화학습은 머신러닝의 세 가지 주요 패러다임입니다.\n모델의 성능은 학습 데이터의 품질과 양에 크게 의존합니다.\n과적합(Overfitting)을 방지하기 위해 정규화 기법이 사용됩니다.\n교차 검증(Cross-validation)은 모델의 일반화 성능을 평가하는 데 유용합니다.\n하이퍼파라미터 튜닝은 모델 최적화의 핵심 단계입니다.',
 '제3장: 머신러닝 기초 개념 - 파트 3\n\n머신러닝의 제3번째 핵심 개념을 다룹니다.\n지도학습, 비지도학습, 강화학습은 머신러닝의 세 가지 주요 패러다임입니다.\n모델의 성능은 학습 데이터의 품질과 양에 크게 의존합니다.\n과적합(Overfitting)을 방지하기 위해 정규화 기법이 사용됩니다.\n교차 검증(Cross-validation)은 모델의 일반화 성능을 평가하는 데 유용합니다.\n하이퍼파라미터 튜닝은 모델 최적화의 핵심 단계입니다.']

In [17]:
# 1024
# 1050
# gpt : llm 

In [23]:
from langchain_huggingface import HuggingFaceEmbeddings

In [24]:
from langchain_openai import OpenAIEmbeddings

In [25]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
text = "임베딩은 텍스트를 벡터로 변환하는 기술입니다"
vector = embeddings.embed_query(text)

In [28]:
len(vector)

384

In [29]:
text = "텍스트 데이터를 벡터로 변환하세요"
vector = embeddings.embed_query(text)

In [30]:
len(vector)

384

In [31]:
documents = [
    "자연어 처리는 인간의 언어를 컴퓨터가 이해하는 기술입니다",
    "임베딩은 텍스트를 벡터로 변환합니다",
    "벡터 검색으로 유사한 문서를 찾을 수 있습니다"]

vectors = embeddings.embed_documents(documents)

In [34]:
len(vectors), len(vectors[0])

(3, 384)

In [35]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [36]:
query = "자연어 처리란 무엇인가요?"
documents = [
    "자연어 처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리하는 기술입니다.",
    "벡터 데이터베이스는 벡터를 저장하고 검색하는 시스템입니다.",
    "딥러닝은 인공신경망을 사용한 머신러닝 방법론입니다.",
    "토큰화는 텍스트를 작은 단위로 분할하는 자연어 처리 기법입니다.",
    "오늘 날씨가 매우 좋습니다.",
]

query_vector = embeddings.embed_query(query)
doc_vectors = embeddings.embed_documents(documents)

In [40]:
similarities = cosine_similarity([query_vector], doc_vectors)[0]

In [41]:
similarities

array([0.4707632 , 0.23142497, 0.24508317, 0.45020823, 0.04189999])

In [42]:
ranked = sorted(enumerate(similarities), key=lambda x : x[1], reverse=True)
ranked

[(0, np.float64(0.47076320166732755)),
 (3, np.float64(0.4502082341928598)),
 (2, np.float64(0.2450831676399254)),
 (1, np.float64(0.23142497215329527)),
 (4, np.float64(0.041899986101878325))]

In [46]:
for rank, (idx, sim) in enumerate(ranked,1):
    print(f" {rank}위 :")
    print(f" {documents[idx]}")

 1위 :
 자연어 처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리하는 기술입니다.
 2위 :
 토큰화는 텍스트를 작은 단위로 분할하는 자연어 처리 기법입니다.
 3위 :
 딥러닝은 인공신경망을 사용한 머신러닝 방법론입니다.
 4위 :
 벡터 데이터베이스는 벡터를 저장하고 검색하는 시스템입니다.
 5위 :
 오늘 날씨가 매우 좋습니다.


In [47]:
sentence_a = "머신러닝은 데이터를 학습하는 알고리즘입니다."
sentence_b = "기계학습은 데이터 기반으로 모델을 훈련시킵니다."

In [48]:
vec_a = embeddings.embed_query(sentence_a)
vec_b = embeddings.embed_query(sentence_b)


In [50]:
similarity = cosine_similarity([vec_a], [vec_b])

In [51]:
similarity

array([[0.86609851]])

In [ ]:
# Vector store
# FAISS
# Chroma DB

In [52]:
sample_docs = [
    Document(page_content="FAISS는 Facebook AI Research가 만든 벡터 검색 라이브러리입니다. 수십억 개의 벡터를 빠르게 검색할 수 있습니다.", metadata={"source": "ai_docs", "category": "vector_db"}),
    Document(page_content="ChromaDB는 오픈소스 임베딩 데이터베이스입니다. Python 네이티브로 간편하게 사용할 수 있습니다.", metadata={"source": "ai_docs", "category": "vector_db"}),
    Document(page_content="RAG(Retrieval-Augmented Generation)는 검색 결과를 활용하여 LLM 답변을 생성하는 기법입니다.", metadata={"source": "ai_docs", "category": "rag"}),
    Document(page_content="LangChain은 LLM 애플리케이션 개발을 위한 프레임워크입니다. 체인, 에이전트, 메모리 등을 제공합니다.", metadata={"source": "ai_docs", "category": "framework"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에서 원하는 출력을 얻기 위해 입력을 설계하는 기술입니다.", metadata={"source": "ai_docs", "category": "prompt"}),
    Document(page_content="벡터 검색에서 코사인 유사도는 두 벡터 간의 각도를 측정합니다. -1에서 1 사이의 값을 가집니다.", metadata={"source": "ai_docs", "category": "similarity"}),
    Document(page_content="MMR(Maximum Marginal Relevance)은 검색 결과의 관련성과 다양성을 동시에 고려하는 알고리즘입니다.", metadata={"source": "ai_docs", "category": "retrieval"}),
    Document(page_content="임베딩은 텍스트를 고차원 벡터로 변환하는 기술입니다. 의미적으로 유사한 텍스트는 가까운 벡터를 가집니다.", metadata={"source": "ai_docs", "category": "embedding"}),
    Document(page_content="BM25는 전통적인 키워드 기반 검색 알고리즘입니다. TF-IDF를 개선한 확률적 검색 모델입니다.", metadata={"source": "ai_docs", "category": "retrieval"}),
    Document(page_content="하이브리드 검색은 키워드 검색과 벡터 검색을 결합하여 검색 품질을 높이는 방법입니다.", metadata={"source": "ai_docs", "category": "retrieval"}),
]

In [53]:
from langchain_community.vectorstores import FAISS as LangFAISS

In [54]:
sample_texts = [
    "LangChain은 LLM 애플리케이션 개발 프레임워크입니다",
    "벡터 스토어는 임베딩을 저장하고 검색하는 데이터베이스입니다",
    "RAG는 검색 증강 생성으로 LLM의 답변 품질을 높입니다",
    "프롬프트 엔지니어링은 AI에게 효과적으로 질문하는 기법입니다",
    "파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 조정합니다",
    "토큰화는 텍스트를 모델이 처리할 수 있는 단위로 나누는 과정입니다",
    "어텐션 메커니즘은 입력의 중요 부분에 집중하는 신경망 기법입니다",
    "트랜스포머는 현대 NLP의 기반이 되는 신경망 아키텍처입니다",
]


In [55]:
metadatas = [
    {"topic": "framework", "level": "beginner"},
    {"topic": "database", "level": "intermediate"},
    {"topic": "technique", "level": "intermediate"},
    {"topic": "technique", "level": "beginner"},
    {"topic": "training", "level": "advanced"},
    {"topic": "preprocessing", "level": "beginner"},
    {"topic": "architecture", "level": "advanced"},
    {"topic": "architecture", "level": "intermediate"},
]

In [56]:
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [58]:
vectorstore = LangFAISS.from_texts(
    texts = sample_texts,
    embedding = embedding_model,
    metadatas = metadatas
)

In [60]:
vectorstore.index.ntotal

8

In [61]:
docs = [
    Document(page_content = text, metadata = meta) for text, meta in zip(sample_texts, metadatas)]

In [62]:
docs

[Document(metadata={'topic': 'framework', 'level': 'beginner'}, page_content='LangChain은 LLM 애플리케이션 개발 프레임워크입니다'),
 Document(metadata={'topic': 'database', 'level': 'intermediate'}, page_content='벡터 스토어는 임베딩을 저장하고 검색하는 데이터베이스입니다'),
 Document(metadata={'topic': 'technique', 'level': 'intermediate'}, page_content='RAG는 검색 증강 생성으로 LLM의 답변 품질을 높입니다'),
 Document(metadata={'topic': 'technique', 'level': 'beginner'}, page_content='프롬프트 엔지니어링은 AI에게 효과적으로 질문하는 기법입니다'),
 Document(metadata={'topic': 'training', 'level': 'advanced'}, page_content='파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 조정합니다'),
 Document(metadata={'topic': 'preprocessing', 'level': 'beginner'}, page_content='토큰화는 텍스트를 모델이 처리할 수 있는 단위로 나누는 과정입니다'),
 Document(metadata={'topic': 'architecture', 'level': 'advanced'}, page_content='어텐션 메커니즘은 입력의 중요 부분에 집중하는 신경망 기법입니다'),
 Document(metadata={'topic': 'architecture', 'level': 'intermediate'}, page_content='트랜스포머는 현대 NLP의 기반이 되는 신경망 아키텍처입니다')]

In [63]:
vectorstore2 = LangFAISS.from_documents(
    documents = docs,
    embedding = embedding_model,
)

In [64]:
vectorstore2.index.ntotal

8

In [65]:
vectorstore2.similarity_search('llm개발 도구', k=3)

[Document(id='36cf6058-650d-4554-81e8-5a4f8721662f', metadata={'topic': 'framework', 'level': 'beginner'}, page_content='LangChain은 LLM 애플리케이션 개발 프레임워크입니다'),
 Document(id='e7ba1c4e-e971-4b49-ada6-374f544b351b', metadata={'topic': 'technique', 'level': 'intermediate'}, page_content='RAG는 검색 증강 생성으로 LLM의 답변 품질을 높입니다'),
 Document(id='61a5bb51-1088-4b64-a7a9-7efdab417ef1', metadata={'topic': 'preprocessing', 'level': 'beginner'}, page_content='토큰화는 텍스트를 모델이 처리할 수 있는 단위로 나누는 과정입니다')]

In [66]:
sample_texts

['LangChain은 LLM 애플리케이션 개발 프레임워크입니다',
 '벡터 스토어는 임베딩을 저장하고 검색하는 데이터베이스입니다',
 'RAG는 검색 증강 생성으로 LLM의 답변 품질을 높입니다',
 '프롬프트 엔지니어링은 AI에게 효과적으로 질문하는 기법입니다',
 '파인튜닝은 사전 학습된 모델을 특정 작업에 맞게 조정합니다',
 '토큰화는 텍스트를 모델이 처리할 수 있는 단위로 나누는 과정입니다',
 '어텐션 메커니즘은 입력의 중요 부분에 집중하는 신경망 기법입니다',
 '트랜스포머는 현대 NLP의 기반이 되는 신경망 아키텍처입니다']

In [83]:
doc_embeddings = embedding_model.embed_documents(sample_texts)
# doc_embeddings

In [68]:
doc_array = np.array(doc_embeddings, dtype = 'float32')
doc_array.shape

(8, 384)

In [70]:
dimension = doc_array.shape[1]
dimension

384

In [72]:
import faiss
index = faiss.IndexFlatL2(dimension)

In [73]:
index.add(doc_array)

In [79]:
query = 'llm개발 도구'
query_emb = np.array([embedding_model.embed_query(query)], dtype='float32')
query_emb.shape

(1, 384)

In [80]:
index.search(query_emb, 3)

(array([[ 8.611778,  8.862035, 10.167585]], dtype=float32), array([[0, 2, 5]]))

In [81]:
distances, indices = index.search(query_emb, 3)

In [82]:
for rank, (dist, ind) in enumerate(zip(distances[0], indices[0])):
    print(f" {rank+1} 위 : [{ind}] {sample_texts[ind]}")

 1 위 : [0] LangChain은 LLM 애플리케이션 개발 프레임워크입니다
 2 위 : [2] RAG는 검색 증강 생성으로 LLM의 답변 품질을 높입니다
 3 위 : [5] 토큰화는 텍스트를 모델이 처리할 수 있는 단위로 나누는 과정입니다


In [84]:
city_docs = [
    "서울의 경복궁은 조선 왕조의 대표적인 궁궐입니다",
    "부산 해운대 해변은 여름 휴양지로 유명합니다",
    "제주도는 한라산과 아름다운 자연경관으로 유명합니다",
    "경주는 신라 시대의 유적지가 많은 역사 도시입니다",
    "전주 한옥마을에서 전통 한식을 즐길 수 있습니다",
    "강릉은 커피 거리와 동해 바다로 유명합니다",
    "인천 차이나타운에서 다양한 중국 음식을 맛볼 수 있습니다",
    "여수 밤바다는 낭만적인 야경으로 인기가 높습니다",
]

In [85]:
documents = [Document(page_content = text) for text in city_docs]
vectorstore = LangFAISS.from_documents(
    documents = documents,
    embedding = embedding_model,
)


In [87]:
def search_cities(query, k):
    results = vectorstore.similarity_search_with_score(query, k)

    for rank, (doc, score) in enumerate(results):
        print(f" {rank+1} 위 : {doc.page_content}") 


In [88]:
search_cities('바다가 보이는 관광지', 3)

 1 위 : 강릉은 커피 거리와 동해 바다로 유명합니다
 2 위 : 부산 해운대 해변은 여름 휴양지로 유명합니다
 3 위 : 여수 밤바다는 낭만적인 야경으로 인기가 높습니다


In [89]:
from langchain_community.vectorstores import Chroma


In [90]:
tech_docs = [
    "GPT-4는 OpenAI의 최신 대규모 언어 모델입니다",
    "BERT는 구글이 개발한 양방향 트랜스포머 모델입니다",
    "LLaMA는 Meta가 공개한 오픈소스 언어 모델입니다",
    "리액트는 페이스북이 만든 프론트엔드 라이브러리입니다",
    "쿠버네티스는 컨테이너 오케스트레이션 도구입니다",
    "도커는 애플리케이션 컨테이너화 플랫폼입니다",
]

metadatas = [
    {"category": "AI", "year": 2023},
    {"category": "AI", "year": 2019},
    {"category": "AI", "year": 2023},
    {"category": "Web", "year": 2013},
    {"category": "DevOps", "year": 2014},
    {"category": "DevOps", "year": 2013},
]


In [91]:
documents = [Document(page_content = text, metadata= meta) for text, meta in zip(tech_docs, metadatas)]
documents

[Document(metadata={'category': 'AI', 'year': 2023}, page_content='GPT-4는 OpenAI의 최신 대규모 언어 모델입니다'),
 Document(metadata={'category': 'AI', 'year': 2019}, page_content='BERT는 구글이 개발한 양방향 트랜스포머 모델입니다'),
 Document(metadata={'category': 'AI', 'year': 2023}, page_content='LLaMA는 Meta가 공개한 오픈소스 언어 모델입니다'),
 Document(metadata={'category': 'Web', 'year': 2013}, page_content='리액트는 페이스북이 만든 프론트엔드 라이브러리입니다'),
 Document(metadata={'category': 'DevOps', 'year': 2014}, page_content='쿠버네티스는 컨테이너 오케스트레이션 도구입니다'),
 Document(metadata={'category': 'DevOps', 'year': 2013}, page_content='도커는 애플리케이션 컨테이너화 플랫폼입니다')]

In [93]:
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = 'tech_articles')

In [94]:
vectorstore.similarity_search_with_score('최신 인공지능 모델', k=3)

[(Document(metadata={'year': 2013, 'category': 'DevOps'}, page_content='도커는 애플리케이션 컨테이너화 플랫폼입니다'),
  12.929512023925781),
 (Document(metadata={'year': 2019, 'category': 'AI'}, page_content='BERT는 구글이 개발한 양방향 트랜스포머 모델입니다'),
  17.312477111816406),
 (Document(metadata={'year': 2023, 'category': 'AI'}, page_content='LLaMA는 Meta가 공개한 오픈소스 언어 모델입니다'),
  20.32349967956543)]

In [95]:
vectorstore.similarity_search_with_score('최신 인공지능 모델', k=3, filter={"category" : "AI"})

[(Document(metadata={'year': 2019, 'category': 'AI'}, page_content='BERT는 구글이 개발한 양방향 트랜스포머 모델입니다'),
  17.312477111816406),
 (Document(metadata={'category': 'AI', 'year': 2023}, page_content='LLaMA는 Meta가 공개한 오픈소스 언어 모델입니다'),
  20.32349967956543),
 (Document(metadata={'category': 'AI', 'year': 2023}, page_content='GPT-4는 OpenAI의 최신 대규모 언어 모델입니다'),
  21.89364242553711)]

In [ ]:
# category : AI
# sub-category : llm, vision, diffusion, ...
# sub-sub-category : gpt, ...

In [ ]:
ChromaDB 이용하셔서 영화추천시스템

In [97]:
movies = [
    ("기생충", "부유한 가족에 기생하는 빈곤 가족의 이야기", "드라마"),
    ("인터스텔라", "우주를 여행하며 인류의 생존을 모색하는 SF 영화", "SF"),
    ("올드보이", "15년간 감금된 남자의 복수 스릴러", "스릴러"),
    ("부산행", "좀비 바이러스가 퍼진 기차 안에서의 생존기", "액션"),
    ("건축학개론", "첫사랑의 추억을 건축으로 풀어낸 로맨스", "로맨스"),
    ("아바타", "외계 행성 판도라에서 벌어지는 모험", "SF"),
    ("범죄도시", "조폭과 형사의 액션 범죄 영화", "액션"),
    ("어바웃타임", "시간 여행으로 사랑을 찾는 로맨스 영화", "로맨스"),
    ("듄", "사막 행성 아라키스에서 펼쳐지는 우주 서사시", "SF"),
    ("해운대", "대형 쓰나미가 부산을 덮치는 재난 영화", "액션"),
]

documents = [Document(page_content = desc, metadata= {'title' : title, 'genre' : genre}) for title, desc, genre in movies]
documents

[Document(metadata={'title': '기생충', 'genre': '드라마'}, page_content='부유한 가족에 기생하는 빈곤 가족의 이야기'),
 Document(metadata={'title': '인터스텔라', 'genre': 'SF'}, page_content='우주를 여행하며 인류의 생존을 모색하는 SF 영화'),
 Document(metadata={'title': '올드보이', 'genre': '스릴러'}, page_content='15년간 감금된 남자의 복수 스릴러'),
 Document(metadata={'title': '부산행', 'genre': '액션'}, page_content='좀비 바이러스가 퍼진 기차 안에서의 생존기'),
 Document(metadata={'title': '건축학개론', 'genre': '로맨스'}, page_content='첫사랑의 추억을 건축으로 풀어낸 로맨스'),
 Document(metadata={'title': '아바타', 'genre': 'SF'}, page_content='외계 행성 판도라에서 벌어지는 모험'),
 Document(metadata={'title': '범죄도시', 'genre': '액션'}, page_content='조폭과 형사의 액션 범죄 영화'),
 Document(metadata={'title': '어바웃타임', 'genre': '로맨스'}, page_content='시간 여행으로 사랑을 찾는 로맨스 영화'),
 Document(metadata={'title': '듄', 'genre': 'SF'}, page_content='사막 행성 아라키스에서 펼쳐지는 우주 서사시'),
 Document(metadata={'title': '해운대', 'genre': '액션'}, page_content='대형 쓰나미가 부산을 덮치는 재난 영화')]

In [98]:
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = 'movies')

In [99]:
def recommend(query, genre=None, k=3):
    if genre:
        results = vectorstore.similarity_search_with_score(query, k=k, filter={"genre":genre})
    else:
        results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"추천 {query}, genre = {genre}")
    for i, (doc, score) in enumerate(results):
        print(f" {i+1} 순위 : {doc}")

In [100]:
recommend('긴장감 넘치는 범죄이야기')

추천 긴장감 넘치는 범죄이야기, genre = None
 1 순위 : page_content='조폭과 형사의 액션 범죄 영화' metadata={'genre': '액션', 'title': '범죄도시'}
 2 순위 : page_content='15년간 감금된 남자의 복수 스릴러' metadata={'title': '올드보이', 'genre': '스릴러'}
 3 순위 : page_content='시간 여행으로 사랑을 찾는 로맨스 영화' metadata={'title': '어바웃타임', 'genre': '로맨스'}


In [101]:
recommend('긴장감 넘치는 범죄이야기', genre='로맨스')

추천 긴장감 넘치는 범죄이야기, genre = 로맨스
 1 순위 : page_content='시간 여행으로 사랑을 찾는 로맨스 영화' metadata={'genre': '로맨스', 'title': '어바웃타임'}
 2 순위 : page_content='첫사랑의 추억을 건축으로 풀어낸 로맨스' metadata={'genre': '로맨스', 'title': '건축학개론'}


In [ ]:
similarity_search
similarity_search_with_score
similarity_search_with_relevance_score

In [103]:
vectorstore.similarity_search_with_relevance_scores('긴장감 넘치는 범죄이야기', k=3)

/tmp/ipykernel_2316011/1025363188.py:1: UserWarning: Relevance scores must be between 0 and 1, got [(Document(metadata={'title': '범죄도시', 'genre': '액션'}, page_content='조폭과 형사의 액션 범죄 영화'), -5.859203359421336), (Document(metadata={'title': '올드보이', 'genre': '스릴러'}, page_content='15년간 감금된 남자의 복수 스릴러'), -8.319962956944924), (Document(metadata={'title': '어바웃타임', 'genre': '로맨스'}, page_content='시간 여행으로 사랑을 찾는 로맨스 영화'), -11.618690917009141)]
  vectorstore.similarity_search_with_relevance_scores('긴장감 넘치는 범죄이야기', k=3)


[(Document(metadata={'title': '범죄도시', 'genre': '액션'}, page_content='조폭과 형사의 액션 범죄 영화'),
  -5.859203359421336),
 (Document(metadata={'title': '올드보이', 'genre': '스릴러'}, page_content='15년간 감금된 남자의 복수 스릴러'),
  -8.319962956944924),
 (Document(metadata={'title': '어바웃타임', 'genre': '로맨스'}, page_content='시간 여행으로 사랑을 찾는 로맨스 영화'),
  -11.618690917009141)]

In [105]:
vectorstore.similarity_search_with_score('긴장감 넘치는 범죄이야기', k=3)

[(Document(metadata={'genre': '액션', 'title': '범죄도시'}, page_content='조폭과 형사의 액션 범죄 영화'),
  9.70037841796875),
 (Document(metadata={'title': '올드보이', 'genre': '스릴러'}, page_content='15년간 감금된 남자의 복수 스릴러'),
  13.180418014526367),
 (Document(metadata={'genre': '로맨스', 'title': '어바웃타임'}, page_content='시간 여행으로 사랑을 찾는 로맨스 영화'),
  17.845523834228516)]

In [110]:
def search_with_threshold(store, query, k=5, threshold=0.5):

    results = store.similarity_search_with_relevance_scores(query, k=k)
    filtered = [(doc, score) for doc, score in results if score >= threshold]

    return filtered

In [111]:
knowledge_base = [
    Document(page_content="파이썬의 리스트 컴프리헨션은 간결한 반복 처리를 가능하게 합니다",
             metadata={"topic": "python", "difficulty": "beginner"}),
    Document(page_content="데코레이터는 함수를 감싸서 추가 기능을 부여하는 파이썬 패턴입니다",
             metadata={"topic": "python", "difficulty": "intermediate"}),
    Document(page_content="async/await를 사용하면 비동기 프로그래밍을 직관적으로 작성할 수 있습니다",
             metadata={"topic": "python", "difficulty": "advanced"}),
    Document(page_content="SQL의 JOIN은 여러 테이블의 데이터를 결합하는 핵심 연산입니다",
             metadata={"topic": "database", "difficulty": "beginner"}),
    Document(page_content="인덱스 최적화는 대용량 데이터베이스 쿼리 성능의 핵심입니다",
             metadata={"topic": "database", "difficulty": "advanced"}),
    Document(page_content="Git 브랜치 전략은 팀 협업의 기본입니다",
             metadata={"topic": "devops", "difficulty": "beginner"}),
    Document(page_content="CI/CD 파이프라인은 코드 배포를 자동화합니다",
             metadata={"topic": "devops", "difficulty": "intermediate"}),
    Document(page_content="컨테이너 오케스트레이션은 대규모 서비스 운영에 필수입니다",
             metadata={"topic": "devops", "difficulty": "advanced"}),
]

kb_store = LangFAISS.from_documents(knowledge_base, embedding_model)

In [116]:
queries = ['파이썬 프로그래밍', '우주 탐사 기술', '데이터베이스 성능']
for i, q in enumerate(queries):
    print(i)
    filtered = kb_store.similarity_search_with_relevance_scores(q, k=3)
    print(filtered)

0
[(Document(id='8a56e2ee-a117-454c-9a43-862b7aee678f', metadata={'topic': 'python', 'difficulty': 'intermediate'}, page_content='데코레이터는 함수를 감싸서 추가 기능을 부여하는 파이썬 패턴입니다'), np.float32(-3.7174888)), (Document(id='3835d47b-8795-41a0-b228-baa5fd35b879', metadata={'topic': 'python', 'difficulty': 'beginner'}, page_content='파이썬의 리스트 컴프리헨션은 간결한 반복 처리를 가능하게 합니다'), np.float32(-5.156187)), (Document(id='67261382-483f-4b88-9b06-ecf60d6cbaeb', metadata={'topic': 'devops', 'difficulty': 'intermediate'}, page_content='CI/CD 파이프라인은 코드 배포를 자동화합니다'), np.float32(-5.334239))]
1
[(Document(id='22aabd2d-e2c7-4931-9da0-45a76f7be315', metadata={'topic': 'devops', 'difficulty': 'advanced'}, page_content='컨테이너 오케스트레이션은 대규모 서비스 운영에 필수입니다'), np.float32(-12.891959)), (Document(id='67261382-483f-4b88-9b06-ecf60d6cbaeb', metadata={'topic': 'devops', 'difficulty': 'intermediate'}, page_content='CI/CD 파이프라인은 코드 배포를 자동화합니다'), np.float32(-12.978985)), (Document(id='8a56e2ee-a117-454c-9a43-862b7aee678f', metadata={'topic'

/tmp/ipykernel_2316011/3448772093.py:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8a56e2ee-a117-454c-9a43-862b7aee678f', metadata={'topic': 'python', 'difficulty': 'intermediate'}, page_content='데코레이터는 함수를 감싸서 추가 기능을 부여하는 파이썬 패턴입니다'), np.float32(-3.7174888)), (Document(id='3835d47b-8795-41a0-b228-baa5fd35b879', metadata={'topic': 'python', 'difficulty': 'beginner'}, page_content='파이썬의 리스트 컴프리헨션은 간결한 반복 처리를 가능하게 합니다'), np.float32(-5.156187)), (Document(id='67261382-483f-4b88-9b06-ecf60d6cbaeb', metadata={'topic': 'devops', 'difficulty': 'intermediate'}, page_content='CI/CD 파이프라인은 코드 배포를 자동화합니다'), np.float32(-5.334239))]
  filtered = kb_store.similarity_search_with_relevance_scores(q, k=3)
/tmp/ipykernel_2316011/3448772093.py:4: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='22aabd2d-e2c7-4931-9da0-45a76f7be315', metadata={'topic': 'devops', 'difficulty': 'advanced'}, page_content='컨테이너 오케스트레이션은 대규모 서비스 운영에 필수입니다'), np.float32(-1

In [112]:
queries = ['파이썬 프로그래밍', '우주 탐사 기술', '데이터베이스 성능']
for q in queries:
    filtered = search_with_threshold(kb_store, q, threshold=0.5)
    print(filtered)

[]
[]
[]


/tmp/ipykernel_2316011/3623532664.py:3: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='8a56e2ee-a117-454c-9a43-862b7aee678f', metadata={'topic': 'python', 'difficulty': 'intermediate'}, page_content='데코레이터는 함수를 감싸서 추가 기능을 부여하는 파이썬 패턴입니다'), np.float32(-3.7174888)), (Document(id='3835d47b-8795-41a0-b228-baa5fd35b879', metadata={'topic': 'python', 'difficulty': 'beginner'}, page_content='파이썬의 리스트 컴프리헨션은 간결한 반복 처리를 가능하게 합니다'), np.float32(-5.156187)), (Document(id='67261382-483f-4b88-9b06-ecf60d6cbaeb', metadata={'topic': 'devops', 'difficulty': 'intermediate'}, page_content='CI/CD 파이프라인은 코드 배포를 자동화합니다'), np.float32(-5.334239)), (Document(id='c08d74d2-cdb6-4f73-a304-8c4b7e8803c1', metadata={'topic': 'python', 'difficulty': 'advanced'}, page_content='async/await를 사용하면 비동기 프로그래밍을 직관적으로 작성할 수 있습니다'), np.float32(-6.152342)), (Document(id='f33a851b-00b7-47a4-a804-82f3f6c7937c', metadata={'topic': 'devops', 'difficulty': 'beginner'}, page_content='Git 브랜치 전략은 팀 협업의 기본입니

In [ ]:
# MMR (Maximum Marginal Relevance)

In [ ]:
# 검색 - 검색결과를 쿼리에 넣어서 - llm.invoke
# 1. 문서 : loader , chunk
# 2. 검색 : 어떤 전략

In [ ]:
# 관련성 - 다양성

In [117]:
mmr_docs = [
    Document(page_content="파이썬으로 웹 크롤링하는 방법: BeautifulSoup 라이브러리 사용",
             metadata={"method": "BeautifulSoup"}),
    Document(page_content="파이썬 웹 스크래핑 기초: requests와 BeautifulSoup 활용",
             metadata={"method": "BeautifulSoup"}),
    Document(page_content="파이썬 크롤링 자동화: Selenium 웹드라이버 사용법",
             metadata={"method": "Selenium"}),
    Document(page_content="Scrapy 프레임워크로 대규모 웹 크롤링 구축하기",
             metadata={"method": "Scrapy"}),
    Document(page_content="파이썬 API 호출로 데이터 수집하기: requests 라이브러리",
             metadata={"method": "API"}),
    Document(page_content="파이썬 데이터 분석: 판다스로 CSV 파일 처리하기",
             metadata={"method": "pandas"}),
    Document(page_content="파이썬 시각화: matplotlib으로 그래프 그리기",
             metadata={"method": "matplotlib"}),
    Document(page_content="파이썬 웹 크롤러: httpx와 BeautifulSoup 비동기 크롤링",
             metadata={"method": "BeautifulSoup"}),
]

In [118]:
mmr_store = LangFAISS.from_documents(mmr_docs, embedding_model)

In [119]:
query = '파이썬으로 웹 데이터 수집'

mmr_store.similarity_search(query, k=4)

[Document(id='4af35ccc-945a-4426-8ca5-185544d2b71f', metadata={'method': 'API'}, page_content='파이썬 API 호출로 데이터 수집하기: requests 라이브러리'),
 Document(id='5e7438d9-c269-4ccb-a55a-e6c59c7400f1', metadata={'method': 'Scrapy'}, page_content='Scrapy 프레임워크로 대규모 웹 크롤링 구축하기'),
 Document(id='1913c93b-0f70-4479-a14b-e7238c6ef7e9', metadata={'method': 'BeautifulSoup'}, page_content='파이썬 웹 스크래핑 기초: requests와 BeautifulSoup 활용'),
 Document(id='a3f2903d-076d-4bdd-8346-5cb527d0fd23', metadata={'method': 'pandas'}, page_content='파이썬 데이터 분석: 판다스로 CSV 파일 처리하기')]

In [120]:
mmr_store.max_marginal_relevance_search(query, k=4, fetch_k=8, lambda_mult=0.5)  # 1:관련성 ----- 0 : 다양성

[Document(id='4af35ccc-945a-4426-8ca5-185544d2b71f', metadata={'method': 'API'}, page_content='파이썬 API 호출로 데이터 수집하기: requests 라이브러리'),
 Document(id='faa6870a-18f6-45b9-9afc-15c04e2719be', metadata={'method': 'matplotlib'}, page_content='파이썬 시각화: matplotlib으로 그래프 그리기'),
 Document(id='5e7438d9-c269-4ccb-a55a-e6c59c7400f1', metadata={'method': 'Scrapy'}, page_content='Scrapy 프레임워크로 대규모 웹 크롤링 구축하기'),
 Document(id='a3f2903d-076d-4bdd-8346-5cb527d0fd23', metadata={'method': 'pandas'}, page_content='파이썬 데이터 분석: 판다스로 CSV 파일 처리하기')]

In [122]:
for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
    result = mmr_store.max_marginal_relevance_search(query, k=4, fetch_k=8, lambda_mult=lam)
    print(result)
    print('------------------------')

[Document(id='4af35ccc-945a-4426-8ca5-185544d2b71f', metadata={'method': 'API'}, page_content='파이썬 API 호출로 데이터 수집하기: requests 라이브러리'), Document(id='faa6870a-18f6-45b9-9afc-15c04e2719be', metadata={'method': 'matplotlib'}, page_content='파이썬 시각화: matplotlib으로 그래프 그리기'), Document(id='1ee1671c-98a3-442e-a8a1-69d034ff65b0', metadata={'method': 'Selenium'}, page_content='파이썬 크롤링 자동화: Selenium 웹드라이버 사용법'), Document(id='1e0dfb3a-c01e-4101-b7ff-bbc700dece30', metadata={'method': 'BeautifulSoup'}, page_content='파이썬 웹 크롤러: httpx와 BeautifulSoup 비동기 크롤링')]
------------------------
[Document(id='4af35ccc-945a-4426-8ca5-185544d2b71f', metadata={'method': 'API'}, page_content='파이썬 API 호출로 데이터 수집하기: requests 라이브러리'), Document(id='faa6870a-18f6-45b9-9afc-15c04e2719be', metadata={'method': 'matplotlib'}, page_content='파이썬 시각화: matplotlib으로 그래프 그리기'), Document(id='5e7438d9-c269-4ccb-a55a-e6c59c7400f1', metadata={'method': 'Scrapy'}, page_content='Scrapy 프레임워크로 대규모 웹 크롤링 구축하기'), Document(id='a3f2903d-076d-

In [123]:
news_docs = [
    # AI 관련
    Document(page_content="OpenAI가 GPT-5 개발 계획을 발표했다", metadata={"subtopic": "AI모델"}),
    Document(page_content="GPT-5의 성능이 기존 대비 크게 향상될 전망이다", metadata={"subtopic": "AI모델"}),
    Document(page_content="구글이 새로운 AI 모델 Gemini Ultra를 공개했다", metadata={"subtopic": "AI모델"}),
    Document(page_content="Meta가 LLaMA 3 오픈소스 모델을 출시했다", metadata={"subtopic": "AI모델"}),
    # AI 규제
    Document(page_content="EU가 AI 규제법을 최종 승인했다", metadata={"subtopic": "AI규제"}),
    Document(page_content="미국 정부가 AI 안전 가이드라인을 발표했다", metadata={"subtopic": "AI규제"}),
    Document(page_content="한국도 AI 기본법 제정을 추진 중이다", metadata={"subtopic": "AI규제"}),
    Document(page_content="AI 윤리 위원회가 규제 프레임워크를 제안했다", metadata={"subtopic": "AI규제"}),
    # AI 산업
    Document(page_content="AI 스타트업 투자가 역대 최고치를 기록했다", metadata={"subtopic": "AI산업"}),
    Document(page_content="기업들의 AI 도입률이 급격히 증가하고 있다", metadata={"subtopic": "AI산업"}),
    Document(page_content="AI 관련 일자리가 전년 대비 50% 증가했다", metadata={"subtopic": "AI산업"}),
    Document(page_content="AI 반도체 시장이 급성장하고 있다", metadata={"subtopic": "AI산업"}),
]

In [124]:
news_store = LangFAISS.from_documents(news_docs, embedding_model)

In [125]:
query = '인공지능 최신 동향'
for lam in [0.0, 0.25, 0.5, 0.75, 1.0]:
    result = news_store.max_marginal_relevance_search(query, k=4, fetch_k=8, lambda_mult=lam)
    print(result)
    print('------------------------')

[Document(id='cafbfb7e-98c8-4f36-a3a2-42f4c38ee8cd', metadata={'subtopic': 'AI산업'}, page_content='기업들의 AI 도입률이 급격히 증가하고 있다'), Document(id='ef2f3484-e919-4253-971c-438a829ebbdf', metadata={'subtopic': 'AI규제'}, page_content='EU가 AI 규제법을 최종 승인했다'), Document(id='ea997f46-e754-4b4a-9245-66c992ed49e5', metadata={'subtopic': 'AI규제'}, page_content='한국도 AI 기본법 제정을 추진 중이다'), Document(id='b0d580f2-71a6-4705-9307-744c9d64f968', metadata={'subtopic': 'AI규제'}, page_content='미국 정부가 AI 안전 가이드라인을 발표했다')]
------------------------
[Document(id='cafbfb7e-98c8-4f36-a3a2-42f4c38ee8cd', metadata={'subtopic': 'AI산업'}, page_content='기업들의 AI 도입률이 급격히 증가하고 있다'), Document(id='c27c101c-9626-49e8-a3dd-8a21e0bd42c2', metadata={'subtopic': 'AI규제'}, page_content='AI 윤리 위원회가 규제 프레임워크를 제안했다'), Document(id='ea997f46-e754-4b4a-9245-66c992ed49e5', metadata={'subtopic': 'AI규제'}, page_content='한국도 AI 기본법 제정을 추진 중이다'), Document(id='89a91e5e-f8b9-4740-92ac-4fd14c115271', metadata={'subtopic': 'AI산업'}, page_content='AI 관련 일자리가 

In [126]:
news_store.save_local('news_faiss_index')

In [128]:
for i in os.listdir('news_faiss_index'):
    
    size = os.path.getsize(os.path.join('news_faiss_index', i))
    print(i, size)

index.pkl 2108
index.faiss 18477


In [130]:
loaded_store = LangFAISS.load_local('news_faiss_index', embedding_model, allow_dangerous_deserialization=True)

In [131]:
loaded_store.index.ntotal

12

In [132]:
texts = ["FAISS는 벡터 검색 라이브러리입니다", "LangChain은 LLM 프레임워크입니다", "임베딩은 텍스트를 벡터로 변환합니다"]
faiss_store = LangFAISS.from_texts(texts, embedding_model)

In [133]:
save_path = './test_faiss_index'
faiss_store.save_local(save_path)

In [135]:
for i in os.listdir(save_path):
    
    size = os.path.getsize(os.path.join(save_path, i))
    print(i, size)

index.pkl 666
index.faiss 4653


In [137]:
loaded_store = LangFAISS.load_local(save_path, embedding_model, allow_dangerous_deserialization=True)
loaded_store.index.ntotal

3

In [ ]:
# prompt | llm | parser
# retriever

In [138]:
retriever = loaded_store.as_retriever(
    search_type='similarity',
    search_kwargs = {'k':3}
)

In [140]:
retriever.search_type, retriever.search_kwargs

('similarity', {'k': 3})

In [141]:
retriever.invoke('FAISS는?')

[Document(id='55575ba9-2974-469e-9223-d6df6a73962d', metadata={}, page_content='FAISS는 벡터 검색 라이브러리입니다'),
 Document(id='709bd3fe-db06-42b1-a236-098668119730', metadata={}, page_content='임베딩은 텍스트를 벡터로 변환합니다'),
 Document(id='f6537e2b-d6e5-4c12-9460-73eee48dd8d0', metadata={}, page_content='LangChain은 LLM 프레임워크입니다')]

In [ ]:
# prompt | retriever | () | llm | parser

In [142]:
mmr_retriever = news_store.as_retriever(
    search_type = 'mmr',
    search_kwargs = {'k':3, 'fetch_k':10, 'lambda_mult':0.5}
)

In [143]:
mmr_retriever.search_type, mmr_retriever.search_kwargs

('mmr', {'k': 3, 'fetch_k': 10, 'lambda_mult': 0.5})

In [144]:
mmr_retriever.invoke('인공지능 최신 동향')

[Document(id='cafbfb7e-98c8-4f36-a3a2-42f4c38ee8cd', metadata={'subtopic': 'AI산업'}, page_content='기업들의 AI 도입률이 급격히 증가하고 있다'),
 Document(id='f38cd0b8-a7e0-4f58-bd41-fd695c26ddaf', metadata={'subtopic': 'AI모델'}, page_content='Meta가 LLaMA 3 오픈소스 모델을 출시했다'),
 Document(id='ea997f46-e754-4b4a-9245-66c992ed49e5', metadata={'subtopic': 'AI규제'}, page_content='한국도 AI 기본법 제정을 추진 중이다')]

In [200]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel
from langchain_core.prompts import PromptTemplate

In [146]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [148]:
prompt = PromptTemplate.from_template(
    """다음 참고 문서를 기반으로 질문에 답변해주세요.

    참고문서:
    {context}

    질문: {question}
    답변:""")

In [152]:
rag_chain = {"context" : mmr_retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()} | prompt

In [153]:
rag_chain.invoke("인공지능 최신 동향")

StringPromptValue(text='다음 참고 문서를 기반으로 질문에 답변해주세요.\n\n    참고문서:\n    기업들의 AI 도입률이 급격히 증가하고 있다\n\nMeta가 LLaMA 3 오픈소스 모델을 출시했다\n\n한국도 AI 기본법 제정을 추진 중이다\n\n    질문: 인공지능 최신 동향\n    답변:')

In [ ]:
news_store 이용해서 retriever 만들고 prompt 체인까지 만들어보세요

In [154]:
retriever = news_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k':2}
)

In [155]:
rag_chain = {"context" : retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()} | prompt

In [156]:
model_names = [
    'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'sentence-transformers/all-MiniLM-L6-v2'
]

In [157]:
query = '자연어 처리란 무엇인가요?'
documents = [
    "자연어 처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리하는 기술입니다.",
    "딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 방법론입니다.",
    "오늘 서울의 날씨는 맑고 기온은 15도입니다.",
]

for model in model_names:
    emb_model = HuggingFaceEmbeddings(model_name=model)
    q_vec = emb_model.embed_query(query)
    doc_vec = emb_model.embed_documents(documents)
    sims = cosine_similarity([q_vec], doc_vec)[0]
    print(f"유사도 결과")
    for i, (doc, sim) in enumerate(zip(documents, sims)):
        print(f" {sim}, {doc}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


유사도 결과
 0.4707631854358765, 자연어 처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리하는 기술입니다.
 0.24637984420847636, 딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 방법론입니다.
 -0.0228852007955899, 오늘 서울의 날씨는 맑고 기온은 15도입니다.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


유사도 결과
 0.437052479823256, 자연어 처리(NLP)는 인간의 언어를 컴퓨터가 이해하고 처리하는 기술입니다.
 0.4739567763471833, 딥러닝은 인공신경망을 여러 층으로 쌓아 학습하는 방법론입니다.
 0.47711780805558385, 오늘 서울의 날씨는 맑고 기온은 15도입니다.


In [ ]:
# 1: 인덱싱 (vector store 만들기)
# -> 문서 파일, 텍스트 -> load -> splitter -> 임베딩 모델 -> vector store 저장
# 2. 검색 (retrieval)
# -> query -> embedding 모델에서 vectorize -> vector store 에서 검색 -> 관련된 chunk를 소환
# 3. 생성
# -> query + chunk -> prompt ->  llm -> 최종답변

In [159]:
from langchain_community.document_loaders import TextLoader

In [162]:
loader = TextLoader('./data/nlp-keywords.txt')
docs = loader.load()

In [164]:
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
splits = splitter.split_documents(docs)

In [166]:
len(splits)

41

In [167]:
embeddings = HuggingFaceEmbeddings(model_name = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
vectorstore = LangFAISS.from_documents(splits, embeddings)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [168]:
retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [169]:
question = '임베딩이란 무엇인가요?'
retriever.invoke(question)

[Document(id='6e3aeb27-c8c9-4aa7-8316-09fe97a986dc', metadata={'source': './data/nlp-keywords.txt'}, page_content='Embedding\n\n정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken'),
 Document(id='f29e4a8f-8ddf-4430-9bc8-205b07af1826', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 인공지능, 자연어 이해, 명령 기반 처리'),
 Document(id='615915ac-5028-4220-a227-92579a1aee3f', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리')]

In [170]:
retrieved_docs = retriever.invoke(question)
context = '\n'.join([doc.page_content for doc in retrieved_docs])
prompt_text = f"""다음 문맥을 참고하여 질문에 답변해주세요.

문맥 : {context}

질문 : {question}
답변."""

In [171]:
prompt_text

'다음 문맥을 참고하여 질문에 답변해주세요.\n\n문맥 : Embedding\n\n정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.\n예시: "사과"라는 단어를 [0.65, -0.23, 0.17]과 같은 벡터로 표현합니다.\n연관키워드: 자연어 처리, 벡터화, 딥러닝\n\nToken\n연관키워드: 인공지능, 자연어 이해, 명령 기반 처리\n연관키워드: 자연어 처리, 딥러닝, 라이브러리\n\n질문 : 임베딩이란 무엇인가요?\n답변.'

In [172]:
llm.invoke(prompt_text)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AIMessage(content='임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다.\n', additional_kwargs={}, response_metadata={}, id='lc_run--019d2896-63c8-7072-b2c1-f98fde9d3e24-0', tool_calls=[], invalid_tool_calls=[])

In [173]:
question = "BERT란 무엇인가요?"
# rag을 구성하셔서 위 질문에 답변을 받아보세요
retrieved_docs = retriever.invoke(question)
context = '\n'.join([doc.page_content for doc in retrieved_docs])
prompt_text = f"""다음 문맥을 참고하여 질문에 답변해주세요.

문맥 : {context}

질문 : {question}
답변."""
llm.invoke(prompt_text)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


AIMessage(content='BERT는 Google에서 개발한 딥러닝 기반의 자연어 처리 모델입니다. 텍스트를 더 작은 단위인 토큰으로 분할하여 분석하고 이해하는 데 특화되어 있습니다. 구문 분석, 텍스트 요약, 질의응답 등 다양한 자연어 처리 작업에 활용됩니다.\n', additional_kwargs={}, response_metadata={}, id='lc_run--019d28a1-d01f-7ed3-8dc3-8343c3d96b06-0', tool_calls=[], invalid_tool_calls=[])

In [178]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template(
    "다음 문서를 참고하여 질문에 답하세요.\n\n{context}\n\n질문: {question}\n답변:")

In [179]:
from langchain_core.output_parsers import StrOutputParser
rag_chain = ({"context" : retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()} 
            | prompt 
            | llm
            | StrOutputParser()
            )


In [180]:
rag_chain.invoke("BERT란 무엇인가요")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'BERT (Bidirectional Encoder Representations from Transformers)는 Google에서 개발한 딥러닝 모델입니다. 자연어 이해에 특화되어 있으며, 텍스트의 의미를 파악하고 문맥을 이해하는 데 매우 효과적입니다. \n\n좀 더 자세히 설명하자면, BERT는 다음과 같은 특징을 가지고 있습니다.\n\n*   **Bidirectional:** 단어의 문맥을 두 방향으로 모두 고려하여 단어의 의미를 파악합니다.\n*   **Transformer 기반:**  Transformer라는 딥러닝 아키텍처를 기반으로 구축되어, 장거리 의존성을 효과적으로 학습할 수 있습니다.\n*   **구문 분석:** 문장의 구문 구조를 분석하여 문장 전체의 의미를 더 정확하게 파악합니다.\n\nBERT는 다양한 자연어 처리 작업에 활용될 수 있으며, 특히 텍스트 분류, 질의 응답, 개체명 인식 등에서 뛰어난 성능을 보여줍니다.'

In [181]:
retriever.invoke("BERT란 무엇인가요")

[Document(id='615915ac-5028-4220-a227-92579a1aee3f', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리'),
 Document(id='f29e4a8f-8ddf-4430-9bc8-205b07af1826', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 인공지능, 자연어 이해, 명령 기반 처리'),
 Document(id='bfc9dec2-692d-45a6-8a99-888f822591cd', metadata={'source': './data/nlp-keywords.txt'}, page_content='Token\n\n정의: 토큰은 텍스트를 더 작은 단위로 분할하는 것을 의미합니다. 이는 일반적으로 단어, 문장, 또는 구절일 수 있습니다.\n예시: 문장 "나는 학교에 간다"를 "나는", "학교에", "간다"로 분할합니다.\n연관키워드: 토큰화, 자연어 처리, 구문 분석\n\nTokenizer')]

In [182]:
finance_loader = TextLoader('data/finance-keywords.txt')
finance_docs = finance_loader.load()

finance_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
finance_splits = finance_splitter.split_documents(finance_docs)

finance_vectorstore = LangFAISS.from_documents(finance_splits, embeddings)

In [186]:
finance_splits[:5]

[Document(metadata={'source': 'data/finance-keywords.txt'}, page_content='S&P 500\n\n정의: S&P 500은 미국 주식 시장에 상장된 500개의 대형 기업의 주가를 종합한 지수입니다. 이는 미국 경제와 주식 시장의 전반적인 상황을 나타내는 주요 지표로 사용됩니다.\n예시: 애플, 마이크로소프트, 아마존과 같은 대형 기술 기업들이 S&P 500에 포함되어 있습니다.\n연관키워드: 주식 시장, 지수, 대형주\n\nMarket Capitalization'),
 Document(metadata={'source': 'data/finance-keywords.txt'}, page_content='Market Capitalization\n\n정의: 시가총액은 회사의 발행 주식 수와 현재 주가를 곱한 값으로, 회사의 전체 가치를 나타냅니다.\n예시: 애플의 시가총액이 2조 달러를 넘어서면서 S&P 500 지수에서 가장 큰 비중을 차지하게 되었습니다.\n연관키워드: 기업 가치, 주식, 투자\n\nDividend\n\n정의: 배당금은 기업이 주주들에게 이익의 일부를 현금으로 지급하는 것을 말합니다.\n예시: 코카콜라는 50년 이상 연속으로 배당금을 인상해온 S&P 500 기업 중 하나입니다.\n연관키워드: 주주 가치, 수익률, 투자 전략'),
 Document(metadata={'source': 'data/finance-keywords.txt'}, page_content='Blue Chip Stocks\n\n정의: 블루칩 주식은 재무적으로 안정적이고 오랜 기간 동안 꾸준한 실적을 보여온 대형 기업의 주식을 의미합니다.\n예시: 존슨앤존슨, 프록터앤갬블과 같은 기업들은 S&P 500에 포함된 대표적인 블루칩 주식입니다.\n연관키워드: 안정적 투자, 대형주, 배당주\n\nSector Rotation'),
 Document(metadata={'source': 'data/finance-keywords.txt'}, page

In [183]:
finance_retriever = finance_vectorstore.as_retriever(search_kwargs={'k':3})

In [188]:
rag_chain = ({"context" : finance_retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()} 
            | prompt 
            | llm
            | StrOutputParser()
            )

def finance_search(query):

    return rag_chain.invoke(query)

In [189]:
finance_search('S&P 500이 뭔가요')

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'S&P 500은 미국 주식 시장에 상장된 500개의 대형 기업의 주가를 종합한 지수입니다.'

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template(
    "다음 문서를 참고하여 질문에 답하세요.\n\n{context}\n\n질문: {question}\n답변:")

In [190]:
def log_chain_input(x):
    print(f"[디버그] context 길이 : {len(x['context'])}")
    print(f"[디버그] question : {x['question']}")
    return x

In [ ]:
rag_chain = ({"context" : finance_retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()} 
            | prompt 
            | llm
            | StrOutputParser()
            )


In [195]:
debug_chain = (
            {"context" : finance_retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()}
            | RunnableLambda(log_chain_input)
            | prompt | llm | StrOutputParser()
            )
    

In [192]:
debug_chain.invoke('시가총액이 뭔가요?')

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[디버그] context 길이 : 804
[디버그] question : 시가총액이 뭔가요?


'시가총액은 회사의 발행 주식 수와 현재 주가를 곱한 값으로, 회사의 전체 가치를 나타냅니다.\n'

In [ ]:
# 검색된 문서의 개수를 출력하고 포맷팅을 같이 해주는 체인을 만들어주세요

In [196]:
def format_and_count(docs):
    print(f"검색된 문서 개수 : {len(docs)}개")
    return format_docs(docs)

prompt = PromptTemplate.from_template(
    "다음 문서를 참고하여 질문에 답하세요.\n\n{context}\n\n질문: {question}\n답변:")

chain = (
            {"context" : retriever | RunnableLambda(format_and_count) , "question" : RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
            )


In [197]:
chain.invoke('LSTM이 뭔가요?')

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


검색된 문서 개수 : 3개


'LSTM은 Long Short-Term Memory의 약자로, 딥러닝 모델입니다. 이는 텍스트 데이터 처리, 특히 시퀀스 데이터(텍스트, 음성 등)에서 장기적인 의존성을 학습하는 데 사용되는 딥러닝 모델입니다.'

In [198]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

prompt = PromptTemplate.from_template(
    """다음 문서를 참고하여 질문에 답하세요.
    
    문맥 : {context}
    
    질문: {question}
    
    답변:""")

In [ ]:
chain = (
            {"context" : retriever | RunnableLambda(format_docs) , "question" : RunnablePassthrough()}
            | prompt | llm | StrOutputParser()
            )


In [203]:
# step1 : 검색 + 질문 동시 처리
setup = RunnableParallel(
    context = retriever,
    question = RunnablePassthrough()
)
# {'context' : ____, 'question' : ____}
# setup['context'], setup['question']

# step2 : 답변 생성 체인
answer_chain = (
    {"context" : lambda x : format_docs(x['context']), 'question' : lambda x : x['question']}
    | prompt | llm | StrOutputParser() )

# step3 : 결합 - 답변 + 출처
rag_with_source = setup | RunnableParallel(answer= answer_chain, sources = lambda x : x['context'])

In [204]:
rag_with_source.invoke("LSTM이 뭔가요")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'answer': "LSTM (Long Short-Term Memory)은 딥러닝 모델의 한 종류로, 특히 자연어 처리 분야에서 널리 사용됩니다. \n\n**LSTM은 다음과 같은 특징을 가지고 있습니다:**\n\n*   **장기 의존성 문제 해결:** RNN (Recurrent Neural Network)의 단점인 '장기 의존성 문제'를 해결하기 위해 개발되었습니다.  RNN은 과거 정보가 현재 처리 과정에 영향을 미치지 못한다는 문제를 해결하기 위해 설계되었습니다.\n*   **게이트 메커니즘:** LSTM은 게이트 메커니즘을 사용하여 정보를 저장하고, 필요할 때만 꺼내어 사용합니다. 이를 통해 과거 정보에 대한 정보를 효과적으로 관리할 수 있습니다.\n*   **역전파 (Backpropagation Through Time, BPTT):**  LSTM은 역전파 과정을 통해 학습하며, 이전 시점의 정보를 활용하여 현재 시점의 예측에 영향을 미칩니다.\n*   **임베딩:** LSTM은 텍스트 데이터를 임베딩 벡터로 변환하여, 의미론적 유사성을 계산하는 데 사용됩니다.\n\n**요약하자면, LSTM은 자연어 처리 분야에서 텍스트 데이터를 효과적으로 처리하고",
 'sources': [Document(id='615915ac-5028-4220-a227-92579a1aee3f', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리'),
  Document(id='f29e4a8f-8ddf-4430-9bc8-205b07af1826', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 인공지능, 자연어 이해, 명령 기반 처리'),
  Document(id='f45673dd-3371-40b5-bf67-a9e9c61a1260', metadata={'source': './data/nlp-keywor

In [205]:
result = rag_with_source.invoke("LSTM이 뭔가요")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [206]:
result['answer']

'LSTM(Long Short-Term Memory)은 딥러닝 모델로, 특히 자연어 처리 분야에서 사용되는 뇌의 신경망 구조입니다. \n\n**LSTM의 주요 특징:**\n\n*   **장기 의존성 문제 해결:** RNN(Recurrent Neural Network)은 과거 정보를 기억하는 데 어려움을 겪는 단일 시퀀스 데이터를 처리하는 데 적합합니다. LSTM은 장기 의존성을 효과적으로 학습하여 긴 문맥을 처리할 수 있도록 설계되었습니다.\n*   **게이트 메커니즘:** LSTM은 게이트(gate)라는 기능을 통해 정보의 중요도를 조절하여 학습 효율성을 높입니다.\n*   **메모리:** LSTM은 메모리 셀을 사용하여 정보를 저장하고, 필요할 때 필요한 정보를 꺼내어 활용합니다.\n*   **다양한 활용:** 텍스트 분류, 기계 번역, 챗봇 등 다양한 자연어 처리 작업에 활용됩니다.\n\n**요약하자면, LSTM은 딥러닝 모델 중 하나로, 특히 자연어 처리에서 장기 의존성을 처리하는 데 효과적인 기술입니다.**\n\n---\n\n더 궁금한 점이 있으시면 질문해주세요.'

In [207]:
result['sources']

[Document(id='615915ac-5028-4220-a227-92579a1aee3f', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리'),
 Document(id='f29e4a8f-8ddf-4430-9bc8-205b07af1826', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 인공지능, 자연어 이해, 명령 기반 처리'),
 Document(id='f45673dd-3371-40b5-bf67-a9e9c61a1260', metadata={'source': './data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 임베딩, 의미론적 유사성\nLLM (Large Language Model)')]

In [ ]:
# transformer 설명해주세요?
# transformer는 ~~~~~입니다.

# 그거 특징이 뭐지?

# transformer의 특징을 3가지로 알려주세요.
# transformer의 특징은 ~~~입니다.

In [ ]:
# context: ~~
# messages = [SystemMessage(content= '당신은 친절한 상담원입니다, ')
#            ]
# messages.append(HumanMessage)
# messages.append(AIMessage)

# llm.invoke(messages)

In [ ]:
# Conversational RAG

In [208]:
from langchain_core.prompts import ChatPromptTemplate

In [209]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [210]:
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", 
     """당신은 NLP 전문가입니다.
     아래 문맥 정보를 활용하여 질문에 답변해주세요.
     이전 대화 내역이 있으면 참고하여 문맥을 이해하세요.

     문맥 :
     {context}"""),
    ("human",
     """이전 대화:
     {history}

     현재 질문: {question}""")
])

In [211]:
conv_prompt.input_variables

['context', 'history', 'question']

In [212]:
chat_history = []

def get_history():
    if not chat_history:
        return "이전 대화 없음"
    return "\n".join(
        f"Q: {q}\nA: {a}" for q, a in chat_history)

In [215]:
conv_chain = (
    {
        'context' : lambda x : format_docs(retriever.invoke(x['question'])),
        'history' : lambda x : x['history'],
        'question' : lambda x : x['question']
    }
    | conv_prompt
    | llm 
    | StrOutputParser()
)

In [216]:
questions =  ['Transformer란 무엇인가요?', '그것은 어떤 매커니즘을 사용하나요?', '그 매커니즘의 정의를 알려주세요']

for q in questions:
    history_text = get_history()
    answer = conv_chain.invoke({'question' : q, 'history' : history_text})
    chat_history.append((q, answer[:200]))

    print(f"Q : {q}")
    print(f"A : {answer}")
    print()

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q : Transformer란 무엇인가요?
A : Transformer는 자연어 처리에서 사용되는 딥러닝 모델로, 주로 번역, 요약, 텍스트 생성 등에 사용됩니다. 이는 Attention 메커니즘을 기반으로 합니다.




Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q : 그것은 어떤 매커니즘을 사용하나요?
A : Transformer는 Attention 메커니즘을 기반으로 합니다.

Q : 그 매커니즘의 정의를 알려주세요
A : Transformer는 자연어 처리에서 사용되는 딥러닝 모델로, Attention 메커니즘을 기반으로 합니다.



In [ ]:
# finance 대화형 RAG 를 만들어주세요

In [217]:
from langchain_community.retrievers import BM25Retriever

In [219]:
loader = TextLoader('data/nlp-keywords.txt')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
splits = splitter.split_documents(docs)
vectorstore = LangFAISS.from_documents(splits, embedding_model)
retriever = vectorstore.as_retriever(search_kwargs={'k':3})


In [221]:
# splits

In [222]:
bm25 = BM25Retriever.from_documents(splits)
bm25.k = 3

In [223]:
bm25_results = bm25.invoke('TF-IDF')

In [224]:
bm25_results

[Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='TF-IDF (Term Frequency-Inverse Document Frequency)'),
 Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning'),
 Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='데이터 마이닝\n\n정의: 데이터 마이닝은 대량의 데이터에서 유용한 정보를 발굴하는 과정입니다. 이는 통계, 머신러닝, 패턴 인식 등의 기술을 활용합니다.\n예시: 소매업체가 고객 구매 데이터를 분석하여 판매 전략을 수립하는 것은 데이터 마이닝의 예입니다.\n연관키워드: 빅데이터, 패턴 인식, 예측 분석\n\n멀티모달 (Multimodal)')]

In [225]:
retriever.invoke('TF-IDF')

[Document(id='11f5274e-da5c-440b-8d17-ab88bb99bd14', metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning'),
 Document(id='5c163235-1c40-4d32-8eed-804a2b82dde0', metadata={'source': 'data/nlp-keywords.txt'}, page_content='TF-IDF (Term Frequency-Inverse Document Frequency)'),
 Document(id='afe4a4df-beed-4334-9396-082cb1f02a44', metadata={'source': 'data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리')]

In [226]:
from langchain_classic.retrievers.ensemble import EnsembleRetriever

In [227]:
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3
vector_retriever = vectorstore.as_retriever(search_kwargs={'k':3})

In [231]:
ensemble_retriever = EnsembleRetriever(
    retrievers = [bm25_retriever, vector_retriever],
    weights = [0.5, 0.5]
) # 하이브리드 서치

In [232]:
query = 'TF-IDF'
results = ensemble_retriever.invoke(query)
results

[Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='TF-IDF (Term Frequency-Inverse Document Frequency)'),
 Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='정의: TF-IDF는 문서 내에서 단어의 중요도를 평가하는 데 사용되는 통계적 척도입니다. 이는 문서 내 단어의 빈도와 전체 문서 집합에서 그 단어의 희소성을 고려합니다.\n예시: 많은 문서에서 자주 등장하지 않는 단어는 높은 TF-IDF 값을 가집니다.\n연관키워드: 자연어 처리, 정보 검색, 데이터 마이닝\n\nDeep Learning'),
 Document(metadata={'source': 'data/nlp-keywords.txt'}, page_content='데이터 마이닝\n\n정의: 데이터 마이닝은 대량의 데이터에서 유용한 정보를 발굴하는 과정입니다. 이는 통계, 머신러닝, 패턴 인식 등의 기술을 활용합니다.\n예시: 소매업체가 고객 구매 데이터를 분석하여 판매 전략을 수립하는 것은 데이터 마이닝의 예입니다.\n연관키워드: 빅데이터, 패턴 인식, 예측 분석\n\n멀티모달 (Multimodal)'),
 Document(id='afe4a4df-beed-4334-9396-082cb1f02a44', metadata={'source': 'data/nlp-keywords.txt'}, page_content='연관키워드: 자연어 처리, 딥러닝, 라이브러리')]

In [ ]:
# Reranking
